# Rydberg noise-model results — intermediate validation

**Research snapshot:** 13 November 2026  
**Status:** Numerical results from a **reduced, physically informed model**; **not** experimental tomography, a measured CZ infidelity, or a SAT/QEC result.

This notebook presents the latest two-atom Lindblad/channel analysis using experimentally reported Rydberg and intermediate-state decay parameters, including the **corrected intermediate-state scattering branches**. It distinguishes modeled population loss, residual Rydberg occupation, and Pauli-twirled weights.

**Main references**
- Evered *et al.*, “High-fidelity parallel entangling gates on a neutral-atom quantum computer,” *Nature* **622**, 268–272 (2023), https://doi.org/10.1038/s41586-023-06481-y (Methods/Extended Data).
- Reiter and Sørensen, “Effective operator formalism for open quantum systems,” *Physical Review A* **85**, 032111 (2012), https://doi.org/10.1103/PhysRevA.85.032111.

**Numerical provenance:** Derived in the 2026-11-13 validation run of the \`rydberg_qutrit\` model. The numerical tables below are a frozen snapshot and plotting cells operate directly on those stored results; **running this notebook alone does not rerun the full 16-dimensional Lindblad channel solver**.

## 1. Physical system and parameters

Each atom has computational states \(|0\rangle,|1\rangle\), a Rydberg state \(|r\rangle\), and an effective absorbing out-of-model state \(|L\rangle\). Thus the **two-atom physical output dimension is 16**; the **computational input dimension is 4**.

The reduced Hamiltonian retains the \(|1\rangle\leftrightarrow|r\rangle\) drive, detuning and Rydberg interaction \(V_{12}n_1^rn_2^r\), with a phase-shaped pulse of duration approximately **0.2641304 μs**. Time is in μs; \(H/\hbar\) is in rad/μs.

| Parameter | Value | Interpretation |
|---|---:|---|
| Atom | \(^{87}\mathrm{Rb}\) | Experiment's platform |
| \(\Omega_{\mathrm{Rydberg}}/2\pi\) | 4.6 MHz | Effective two-photon coupling |
| Atom separation | 2 μm | Two-atom example |
| \(V_{12}/2\pi\) | ≈450 MHz | Blockade interaction (modeled) |
| Rydberg lifetime \(T_r\) | 88 μs | Experimentally motivated model parameter |
| Intermediate-state lifetime \(T_e\) | 0.110 μs | Experimentally motivated model parameter |
| Single-photon Rabi frequencies | 237 MHz, 303 MHz | Blue/red laser couplings |
| Intermediate-state detuning | 7800 MHz | Large detuning |
| \(T_2^*\) | 3 μs | Used in an **assumed Gaussian common-mode detuning model** |

Rydberg decay branches: \((r\to0,r\to1,r\to L)=(0.053,0.053,0.894)\).

**Corrected intermediate scattering branches from Evered et al.:**

\[
(e\to0,\; e\to1,\; e\to L)=(0.1354,\;0.2504,\;0.6142).
\]

These percentages are *conditional destinations if a scattering event occurs*, **not** probabilities that scattering occurs during one pulse.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rydberg_branches = {"r_to_0": 0.053, "r_to_1": 0.053, "r_to_L": 0.894}
intermediate_branches = {"e_to_0": 0.1354, "e_to_1": 0.2504, "e_to_L": 0.6142}
parameters = {"pulse_us": 1.215/4.6, "T_r_us": 88.0, "T_e_us": .110,
              "T2star_us": 3.0, "Omega_r_MHz": 4.6, "Delta_MHz": 7800.0}
assert np.isclose(sum(rydberg_branches.values()), 1.0)
assert np.isclose(sum(intermediate_branches.values()), 1.0)
print("Pulse duration:", parameters["pulse_us"], "µs")
print("Rydberg loss rate:", rydberg_branches["r_to_L"]/parameters["T_r_us"], "µs⁻¹")
print("Intermediate scattering loss fraction:", intermediate_branches["e_to_L"])

## 2. Channel construction and what its probabilities mean

For collapse operators \(C_{i,a}^{(r)}=\sqrt{\Gamma_r b_a^{(r)}}|a_i\rangle\langle r_i|\), the master equation is

\[
\dot\rho=-i[H(t)/\hbar,\rho]+\sum_{i,a}\mathcal D[C_{i,a}^{(r)}]\rho
+\sum_{i,a}\mathcal D[C_{i,a}^{(e)}(t)]\rho.
\]

In the reduced model, the eliminated intermediate state contributes effective scattering jumps

\[
C_{i,a}^{(e)}(t)=\sqrt{\Gamma_e b_a^{(e)}}\;|a_i\rangle
\frac{\Omega_b e^{-i\phi(t)}\langle1_i|+\Omega_{\mathrm{red}}\langle r_i|}{2\Delta}.
\]

The \(T_2^*\) contribution is a **classical mixture of channels over Gaussian detuning offsets**, not an additional Markovian collapse operator. The common-mode correlation is a modeling assumption.

To construct a channel independent of a chosen \(\rho(0)\), evolve the **16 computational operator-basis elements** \(|a\rangle\langle b|\) through the two-atom physical dynamics. This yields a **64×64 Choi matrix** and, in this numerical reconstruction, **25 nonzero Kraus operators**, each of shape **16×4**.

The Pauli weights for computational output are obtained after removing a selected noiseless reference operation \(U_{\rm ref}\):

\[
q_P=\frac1{16}\sum_j\left|\operatorname{Tr}
\left(P^\dagger U_{\rm ref}^\dagger \Pi_{\rm comp}K_j\right)\right|^2,
\quad P\in\{I,X,Y,Z\}^{\otimes 2}.
\]

These are **Pauli-twirled channel weights** (not literal observed Pauli jumps); the sum \(\sum_Pq_P\) equals the computational-survival probability for a maximally mixed computational input. The leftover weight is final population in residual Rydberg or loss levels. A finite \(|r\rangle\) population at pulse end is an unwanted out-of-computational-space outcome, even though transient \(|r\rangle\) excitation during the pulse is intentional.

## 3. Experiment A — successive noise-model assumptions

All values are **percent of the input-averaged channel**, *not* percentages conditional on an error. The previous all-return-to-\(|1\rangle\) approximation used a 112.8 ns lifetime, whereas the published-branch calculation uses 110 ns; therefore the comparison is **not strictly branching-only**.

| Outcome | Decay + \(T_2^*\) | Prior \(e\to1\) approximation | Published scattering branches |
|---|---:|---:|---:|
| \(II\) Pauli weight | 99.790875% | 99.763852% | **99.757083%** |
| Other Pauli weights | 0.082488% | 0.093463% | **0.089403%** |
| Residual Rydberg | 0.023364% | 0.039409% | **0.028215%** |
| Final loss manifold | 0.103273% | 0.103275% | **0.125299%** |
| Total | ≈100% | ≈100% | ≈100% |

**Finding:** including the measured scattering branches increases modeled final loss and shifts the distribution among computational, Rydberg, and loss outcomes. It does not independently establish agreement with measured gate fidelity.

In [ ]:
models = {
    "Decay + T2*": {"II":99.790875, "Other Paulis":.082488, "Rydberg":.023364, "Loss":.103273},
    "Earlier e→1": {"II":99.763852, "Other Paulis":.093463, "Rydberg":.039409, "Loss":.103275},
    "Published branches": {"II":99.757083, "Other Paulis":.089403, "Rydberg":.028215, "Loss":.125299},
}
fig, axes = plt.subplots(1, 2, figsize=(12,4), constrained_layout=True)
labels = list(models)
for component in ("Other Paulis", "Rydberg", "Loss"):
    axes[0].plot(labels, [models[m][component] for m in labels], 'o-', label=component)
axes[0].set_ylabel("Percentage of computational-input-averaged channel (%)")
axes[0].set_title("Physical outcomes (identity excluded)")
axes[0].legend()
axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(labels, [models[m]["Loss"] for m in labels])
axes[1].set_ylabel("Final loss-state population (%)")
axes[1].set_title("Revised branching increases loss")
axes[1].tick_params(axis='x', rotation=20)
plt.show()

## 4. The Pauli-twirled distribution from the revised model

This is the full numerical Pauli distribution at the final pulse time, after removal of the chosen reference unitary. The \(ZZ\) weight is especially sensitive to the **assumed correlated detuning offsets**; it must not be described as an experimentally measured \(ZZ\) correlation.

**Do not renormalize only the nonidentity weights to sum to one** when comparing against loss. The full outcome accounting uses \(II\), all other Paulis, residual Rydberg and loss.

In [ ]:
pauli = {
"II":.9975708285492755, "IX":1.1213229015286904e-05,
"IY":1.1213229015287602e-05, "IZ":.0002769255426770645,
"XI":1.1213229015278273e-05, "XX":9.709195227125537e-10,
"XY":9.709195227119345e-10, "XZ":1.1199685157723887e-05,
"YI":1.1213229015280748e-05, "YX":9.709195227124565e-10,
"YY":9.70919522712018e-10, "YZ":1.1199685157721824e-05,
"ZI":.00027692554267697375, "ZX":1.1199685157730233e-05,
"ZY":1.1199685157730875e-05, "ZZ":.0002505270922979141,
}
outside = {"Residual Rydberg":.00028214908245120606,
           "Any loss":.0012529886502508048}
assert np.isclose(sum(pauli.values()) + sum(outside.values()), 1.0, atol=3e-10)
assert np.isclose(sum(v for k,v in pauli.items() if k!='II'), .0008940337180220835)
print(f"II = {pauli['II']*100:.6f}%")
print(f"All other Pauli weights = {100*(sum(pauli.values())-pauli['II']):.6f}%")
print(f"Residual Rydberg = {100*outside['Residual Rydberg']:.6f}%")
print(f"Final loss = {100*outside['Any loss']:.6f}%")
print(f"Total = {100*(sum(pauli.values())+sum(outside.values())):.9f}%")

labels = [k for k in pauli if k!='II']
values = [pauli[k]*100 for k in labels]
plt.figure(figsize=(12,4))
plt.bar(labels, values)
plt.yscale('log')
plt.ylabel('Pauli-twirled weight (%) — log scale')
plt.title('Revised two-atom model: nonidentity Pauli weights')
plt.show()

## 5. Experiment B — identify physical scattering/loss pathways

The **expected number** of jumps in channel \(j\) is obtained by integrating its instantaneous jump rate:

\[
\mathbb E[N_j]=\int_0^\tau\operatorname{Tr}
[C_j^\dagger(t) C_j(t)\rho(t)]\,dt.
\]

These expected counts are **not mutually exclusive event probabilities**: more than one event may occur. For two atoms, the expected number of atoms lost can exceed the probability of *at least one* loss because both atoms may be lost.

The following source decomposition is averaged over a maximally mixed computational input.

In [ ]:
jumps = {
  "r → 0": 6.123215893538159e-05,
  "r → 1": 6.123215893538159e-05,
  "r → L": 0.0010328594356270033,
  "e → 0": 4.8615464731405615e-05,
  "e → 1": 8.990629519013272e-05,
  "e → L": 0.00022052893971956657,
}
expected_lost = jumps["r → L"] + jumps["e → L"]
any_loss = .0012529886502513467
both_lost = 3.99725095223538e-07
assert np.isclose(expected_lost, any_loss + both_lost, atol=1e-11)
print("Expected jump counts per pulse (%)")
for k,v in jumps.items(): print(f"  {k:7s}: {100*v:.6f}%")
print(f"Expected number lost = {expected_lost:.9g}")
print(f"P(any loss)          = {any_loss:.9g}")
print(f"P(both lost)         = {both_lost:.9g}")
plt.figure(figsize=(9,3.5))
plt.bar(["Rydberg → L", "Intermediate → L"],
        [100*jumps["r → L"],100*jumps["e → L"]])
plt.ylabel("Expected loss jumps per pulse (%)")
plt.title("Source-resolved loss budget")
plt.show()

## 6. Experiment C — explicitly include the intermediate level

A separate **single-atom** comparison contrasts a five-state model \(\{|0\rangle,|1\rangle,|r\rangle,|L\rangle,|e\rangle\}\) with a four-state adiabatically eliminated approximation. Both use constant, abruptly switched-on laser amplitudes for 0.2641304 μs; **this is a stress test, not the shaped two-atom gate**.

To leading order at large \(|\Delta|\), the intermediate amplitude is

\[
c_e\simeq\frac{\Omega_b c_1+\Omega_{\mathrm{red}}c_r}{2\Delta}.
\]

| Quantity | Explicit \(|e\rangle\) | Effective eliminated \(|e\rangle\) |
|---|---:|---:|
| Final \(|L\rangle\) | **0.173030%** | 0.160311% |
| Final \(|r\rangle\) | 39.05724% | 39.28172% |
| Final \(|e\rangle\) | 0.044707% | Not represented |
| Expected intermediate scattering | **0.091433%** | 0.070665% |

The effective approximation undercounts scattering by ~23% relative to the explicit model in this abrupt-drive example. It should **not** be presented as a 23% error for the actual shaped two-atom pulse.

In [ ]:
explicit = {"Loss": .17303027565506565, "Rydberg": 39.05723635637325,
            "Intermediate": .04470659278670191, "Scattering count": .09143323186933555}
effective = {"Loss": .16031063674245933, "Rydberg": 39.28172036681641,
             "Intermediate": np.nan, "Scattering count": .07066461565106029}
print("Relative scattering underestimation:",
      f"{100*(explicit['Scattering count']-effective['Scattering count'])/explicit['Scattering count']:.2f}%")
fig, ax = plt.subplots(figsize=(9,3.5))
keys=["Loss","Scattering count"]
x=np.arange(len(keys))
ax.bar(x-.18,[explicit[k] for k in keys],width=.36,label="Explicit 5-state")
ax.bar(x+.18,[effective[k] for k in keys],width=.36,label="Effective 4-state")
ax.set_xticks(x,keys)
ax.set_ylabel("Percent (%)")
ax.legend()
ax.set_title("Abrupt-drive single-atom benchmark")
plt.show()

## 7. Consistency checks and limitations

For the **revised two-atom** model:

- Choi dimension: **64×64**; retained Choi-eigenmode Kraus operators: **25**, each **16×4**.
- Reported trace-preservation error: \(8.9\times10^{-16}\).
- Minimum Choi eigenvalue: \(-1.2\times10^{-15}\), consistent with numerical roundoff.
- \(\sum_P q_P + P(\text{residual }r) + P(\text{any }L)=1\) within numerical precision.
- Expected lost-atom count from integrated jumps equals final \(\langle N_L\rangle\).

**Important open issues:** the original experiment notes a multilevel \(4/3\) correction to a first-order scattering estimate; the current surrogate uses one virtual \(|e\rangle\) and does not fully reproduce that correction or all atomic levels. The shared Gaussian detuning model is assumed rather than measured as a two-atom correlation. Additional control/measurement errors are excluded. The reference unitary is chosen from the reduced noiseless computational evolution; this is not a calibrated gate-fidelity measurement. Loss/erasure flagging is not yet modeled.

**Interpretation:** the data support a consistent, explicitly approximate physics-to-channel-to-Pauli workflow, *not yet* a demonstrated hardware-aware QEC advantage.

## 8. Relevance to Special-curriculum

These results are **intermediate supporting calculations**, not yet part of the mathematical SAT result. Their role is to motivate an experimentally informed nonuniform error set.

The SAT paper can remain centered on stabilizer constraints, Hamiltonian symmetries and error-set selection. Incorporating likelihoods or leakage-aware constraints is future work. We should **not** claim we have already shown a lower logical-error rate or a hardware-informed pseudothreshold.

The full revised numerical solver and matrix exports were packaged separately in the 2026-11-13 validation archive. The earlier \`rydberg_qutrit/\` source directory in this repository contains predecessor prototypes; do not confuse their older Kraus dataset with the revised published-branch validation described here.